In [ ]:
import os
from arize.experimental.datasets import ArizeDatasetsClient
from dotenv import load_dotenv

load_dotenv()

HOST = os.getenv("ARIZE_GRPC_HOST")
API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")

client = ArizeDatasetsClient(
    api_key=API_KEY,
    host=HOST
)

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("cais/mmlu", "high_school_psychology")

In [ ]:
df = ds["test"].to_pandas()
df.head()

In [ ]:
from arize.experimental.datasets.utils.constants import GENERATIVE

dataset = client.create_dataset(
    space_id=SPACE_ID, 
    dataset_name="mmlu_high_school_psychology",
    dataset_type=GENERATIVE,
    data=df
)

dataset

In [ ]:
# Define your task
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    # model="gpt-5.1",
    temperature=0,  # 생성적 응답의 다양성
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

prompt = ChatPromptTemplate.from_template("""
You are an expert in answering questions.

Find the correct answer to the question from the choices.

Your response must contain ONLY the index number of the correct answer. Index starts from 0.

Do not include any explanatory text, labels, phrases like "The correct answer is:", or additional words. Output only the single digit.

Question: {input}

Choices: {choices}
"""
)

chain = prompt | llm | StrOutputParser()

def task(dataset_row) -> str:
    response = chain.invoke({"input": dataset_row["question"], "choices": dataset_row["choices"]})
    return response

In [ ]:
task(df.iloc[1])

In [ ]:
# Evaluator 정의하기
from typing import Any, Optional, Mapping
from arize.experimental.datasets.experiments.evaluators.base import (
    EvaluationResult,
    CodeEvaluator,
    JSONSerializable,
    TaskOutput
)

class MMLUEvaluator(CodeEvaluator):
    def evaluate(
        self,
        *,
        dataset_row: Optional[Mapping[str, JSONSerializable]] = None,
        output: Optional[TaskOutput] = None,
        **_: Any,
    ) -> EvaluationResult:
        answer = dataset_row.get("answer") if dataset_row else None # 0, 1, 2, 3
        is_good = int(answer) == int(output)
        label = "correct" if is_good else "incorrect"
        
        return EvaluationResult(
            score=float(is_good),
            label=label,
            explanation=f"answer: {answer}, output: {output}"
        )


In [ ]:
client.run_experiment(
    space_id=SPACE_ID, 
    dataset_name="mmlu_high_school_psychology",
    dataset_df=df,
    task=task, 
    evaluators=[MMLUEvaluator()], #include your evaluation functions here 
    experiment_name="MMLU-Experiment-gpt-4o-mini",
    concurrency=10,
    dry_run=False
)